In [2]:
from langchain_neo4j import GraphCypherQAChain, Neo4jGraph
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
import os

load_dotenv(override=True)

MODEL = "gpt-5.4-mini"

graph = Neo4jGraph(
    url=os.getenv('NEO4J_URI'),
    username=os.getenv('NEO4J_USERNAME'),
    password=os.getenv('NEO4J_PASSWORD'),
    database=os.getenv('NEO4J_DATABASE')
)

llm = ChatOpenAI(model=MODEL)

CYPHER_PROMPT = PromptTemplate(
    input_variables=["schema", "question"],
    template="""당신은 Neo4j Cypher 쿼리 전문가입니다.
아래 스키마를 참고하여 질문에 맞는 Cypher 쿼리를 생성하세요.

스키마:
{schema}

관계 방향 규칙 (반드시 지켜야 함):
- (:치매안심센터)-[:LOCATED_IN]->(:시군구)
- (:시군구)-[:CONTAINS]->(:시도)

올바른 예시:
MATCH (c:치매안심센터)-[:LOCATED_IN]->(sg:시군구)-[:CONTAINS]->(sd:시도 {{시도명: '서울특별시'}})
WHERE sg.시군구명 = '강남구'
RETURN c.센터명, c.주소, c.전화번호

질문: {question}
Cypher 쿼리:"""
)

chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    allow_dangerous_requests=True,
    verbose=True,
    cypher_prompt=CYPHER_PROMPT
)

test_queries = [
    "서울에 치매안심센터 몇 개야?",
    "경기도 수원시 치매안심센터 알려줘",
    "치매안심센터가 가장 많은 시도는?",
    "제주도 치매안심센터 전화번호 알려줘",
    "2018년에 설립된 서울 치매안심센터는?"
]

for query in test_queries:
    print(f"\nQ: {query}")
    result = chain.invoke({"query": query})
    print(f"A: {result['result']}")
    print("---")


Q: 서울에 치매안심센터 몇 개야?


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (c:치매안심센터)-[:LOCATED_IN]->(sg:시군구)-[:CONTAINS]->(sd:시도 {시도명: '서울특별시'})
RETURN count(c) AS 치매안심센터수

Full Context:
[{'치매안심센터수': 31}]

> Finished chain.
A: 서울에 치매안심센터는 31개입니다.
---

Q: 경기도 수원시 치매안심센터 알려줘


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (c:치매안심센터)-[:LOCATED_IN]->(sg:시군구)-[:CONTAINS]->(sd:시도 {시도명: '경기도'})
WHERE sg.시군구명 = '수원시'
RETURN c.센터명, c.주소, c.전화번호, c.위도, c.경도, c.홈페이지

Full Context:
[]

> Finished chain.
A: 죄송하지만, 해당 정보만으로는 경기도 수원시 치매안심센터에 대해 알 수 없습니다.
---

Q: 치매안심센터가 가장 많은 시도는?


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (c:치매안심센터)-[:LOCATED_IN]->(sg:시군구)-[:CONTAINS]->(sd:시도)
RETURN sd.시도명 AS 시도명, COUNT(c) AS 치매안심센터수
ORDER BY 치매안심센터수 DESC
LIMIT 1

Full Context:
[{'시도명': '경기도', '치매안심센터수': 46}]

> Finished chain.
A: 경기도입니다.
---

Q: 제주도 치매안심센터 전화번호 알려줘


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (c:치매안심센터)-[:LOCATE